# Example: Wind-Only Analysis Species — Bobolink (*Dolichonyx oryzivorus*)

This notebook demonstrates the **Manuscript 2 wind-only analysis pipeline** applied to Bobolink breeding season data. It serves as a worked example illustrating model selection and GAM-based abundance modelling for a grassland bird species in relation to wind energy infrastructure.

**Species:** Bobolink (*Dolichonyx oryzivorus*)  
**Season:** Breeding (May–July)  
**Spatial grain:** 1.5 km hexagonal grid  
**Study period:** 2012–2021  
**Response variable:** Rounded mean eBird count per hexagon-year (`meanCount_round`)  
**Key finding:** `s(WindCount, k=5)` selected as best wind variable; negative binomial GAM with full nuisance structure

---

**Notebook structure:**
1. Setup & library loading
2. Data loading & joining
3. Prevalence & wind exposure summary
4. Wind categorical variable creation
5. Drop NAs & final prevalence
6. Correlation screening
7. Scaling & train/test split
8. Exploratory data analysis
9. GAM family & null model
10. Wind variable selection
11. Best model summary & diagnostics
12. Prediction & visualization

---
*ecological_data_science - Samantha McGarrigle*

## 1. Setup

In [ ]:
library(dplyr)
library(tidyr)
library(ggplot2)
library(mgcv)
library(MuMIn)
library(corrplot)

# Resolve namespace conflicts
select <- dplyr::select
gam    <- mgcv::gam

# GAM smoothing parameters
k      <- 5  # basis dimension for most smooths
k_time <- 7  # basis dimension for cyclic time-of-day smooth

# Knot positions for meanStartTime cyclic spline
# Ensures the spline joins correctly at midnight even if no data fall near midnight
time_knots <- list(meanStartTime = seq(0, 24, length.out = k_time))

# Timing helper: wraps gam() and prints elapsed time
timed_gam <- function(formula, data, family, knots) {
  t0 <- Sys.time()
  m  <- gam(formula, data = data, family = family, knots = knots)
  cat("Elapsed:", round(Sys.time() - t0, 2), "\n")
  m
}

## 2. Data Loading & Joining

Two files are required:
- `species_season_finaldata.csv` — eBird hexagon-year data for Bobolink breeding season (90,881 rows x 45 cols)
- `WRD_foranalysis.csv` — Wind turbine attributes joined by `cell_year`

This is a **wind-only** analysis: no CRP file is loaded.

In [ ]:
turbines <- read.csv(file.choose(), header = TRUE)  # WRD_foranalysis.csv
summary(turbines)

In [ ]:
boboli_breed1_1.5  <- read.csv(file.choose())              # species_season_finaldata.csv
boboli_breed1b_1.5 <- boboli_breed1_1.5[, c(1:11, 36:45)] # retain effort, land cover, and response cols

In [ ]:
# Join turbine attributes by cell_year; fill WindCount = 0 where no turbines are present
boboli_breed_1.5 <- left_join(boboli_breed1b_1.5, turbines, by = "cell_year") %>%
  mutate(WindCount = ifelse(is.na(WindCount), 0, WindCount))

summary(boboli_breed_1.5)

## 3. Prevalence & Wind Exposure Summary

In [ ]:
# Detection prevalence
count(boboli_breed_1.5, species_observed) %>%
  mutate(percent = n / sum(n))

# Proportion of hexagon-years with at least one turbine
count(boboli_breed_1.5, wind_present = WindCount > 0) %>%
  mutate(percent = n / sum(n))

## 4. Wind Categorical Variable Creation

Wind turbine attributes are recoded into ordered factor bins used as factors in GAMs.
All categories include a `"None"` level for hexagons with no turbines.
A helper function handles the shared binning logic to avoid code repetition.

In [ ]:
# Helper: bin a continuous wind attribute into ordered categories,
# assigning "None" where no turbine is present
wind_bin <- function(x, pa, breaks, labels) {
  ifelse(pa == "N", "None",
         as.character(cut(x, breaks = breaks, labels = labels, right = TRUE)))
}

boboli_breed_1.5 <- boboli_breed_1.5 %>%
  mutate(
    WindPA = factor(ifelse(WindCount > 0, "Y", "N")),

    WindAge_cat = factor(
      wind_bin(WindAge, WindPA,
               breaks = c(-Inf, 2, 4, 6, 8, Inf),
               labels = c("0-2", "3-4", "5-6", "7-8", "9+")),
      levels = c("None", "0-2", "3-4", "5-6", "7-8", "9+")),

    WindHeight_cat = factor(
      wind_bin(WindHeight, WindPA,
               breaks = c(0, 20, 40, 60, 80, Inf),
               labels = c("1-20", "21-40", "41-60", "61-80", "81-100+")),
      levels = c("None", "1-20", "21-40", "41-60", "61-80", "81-100+")),

    WindRSA_cat = factor(
      wind_bin(WindRSA, WindPA,
               breaks = c(0, 5000, 10000, 15000, Inf),
               labels = c("0-5000", "5001-10000", "10001-15000", "15000+")),
      levels = c("None", "0-5000", "5001-10000", "10001-15000", "15000+")),

    WindCap_cat = factor(
      wind_bin(WindCap, WindPA,
               breaks = c(0, 1000, 2000, 3000, 4000, Inf),
               labels = c("0-1000", "1001-2000", "2001-3000", "3001-4000", "4001+")),
      levels = c("None", "0-1000", "1001-2000", "2001-3000", "3001-4000", "4001+"))
  )

summary(select(boboli_breed_1.5, WindPA, WindAge_cat, WindHeight_cat, WindRSA_cat, WindCap_cat))

## 5. Drop NAs & Final Prevalence

Rows where wind attribute categories are `NA` arise from turbines with missing attribute data in the WRD. These are dropped prior to modelling.

In [ ]:
boboli_breed_1.5_NA <- boboli_breed_1.5 %>%
  drop_na(WindHeight_cat, WindCap_cat, WindRSA_cat)

str(boboli_breed_1.5_NA)

# Final prevalence and wind exposure after NA removal
count(boboli_breed_1.5_NA, species_observed) %>% mutate(percent = n / sum(n))
count(boboli_breed_1.5_NA, wind_present = WindCount > 0) %>% mutate(percent = n / sum(n))

## 6. Correlation Screening

Pairwise Pearson correlations among land cover and effort covariates are examined prior to model building.
A conservative threshold of |r| > 0.4 is used to flag collinear pairs.

**Result:** Cropland and TreeCover were negatively correlated; Cropland was retained based on higher deviance explained in univariate models.

**Final land cover set:** Developed, Cropland, GrassShrub, Water, Wetland, Barren, IceSnow (linear)

In [ ]:
corr_vars <- c("WindCount",
               "Developed", "Cropland", "GrassShrub", "TreeCover",
               "Water", "Wetland", "Barren", "IceSnow",
               "meanStartTime", "meanSampDur", "meanDIST", "meanNumObs")

corr_matrix <- cor(select(boboli_breed_1.5_NA, all_of(corr_vars)), use = "complete.obs")

options(repr.plot.width = 14, repr.plot.height = 14)
corrplot(corr_matrix, method = "circle", col = COL2("PiYG", 10),
         addCoef.col = "black", insig = "pch", pch = c("", "."),
         sig.level = 0.4)

## 7. Scaling & Train/Test Split

All continuous predictors are standardized (mean = 0, SD = 1) prior to modelling.
Scaling parameters are saved for back-transformation of predictions.

In [ ]:
scale_vars <- c("meanStartTime", "meanSampDur", "meanDIST", "meanNumObs",
                "Developed", "Cropland", "GrassShrub", "TreeCover",
                "Water", "Wetland", "Barren", "IceSnow",
                "WindCount", "Lat", "Long")

# Save means and SDs for back-transformation
scaling_params <- boboli_breed_1.5_NA %>%
  select(all_of(scale_vars)) %>%
  summarise(across(everything(),
    list(mean = ~ mean(., na.rm = TRUE), sd = ~ sd(., na.rm = TRUE)))) %>%
  pivot_longer(everything(), names_to = c("variable", "statistic"),
               names_pattern = "(.*)_(mean|sd)") %>%
  pivot_wider(names_from = "statistic", values_from = "value")

scaling_params

In [ ]:
# Standardize, rename categorical wind cols, select modelling columns, split 80/20
boboli_breed_split <- boboli_breed_1.5_NA %>%
  mutate(across(all_of(scale_vars), scale)) %>%
  select(meanCount, meanCount_round,
         all_of(scale_vars),
         WindPA,
         WindHeight = WindHeight_cat,
         WindAge    = WindAge_cat,
         WindRSA    = WindRSA_cat,
         WindCap    = WindCap_cat) %>%
  split(if_else(runif(nrow(.)) <= 0.8, "train", "test"))

cat("Train:", nrow(boboli_breed_split$train),
    " Test:", nrow(boboli_breed_split$test), "\n")

## 8. Exploratory Data Analysis

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 5)

par(mfrow = c(1, 2))
hist(boboli_breed_1.5_NA$meanCount,
     main = "All counts", xlab = "Observed count")
hist(boboli_breed_1.5_NA$meanCount[boboli_breed_1.5_NA$meanCount > 0],
     main = "Non-zero counts", xlab = "Observed count > 0")
par(mfrow = c(1, 1))

In [ ]:
cat("Zeros:    ", sum(boboli_breed_1.5_NA$meanCount_round == 0), "\n")
cat("Non-zeros:", sum(boboli_breed_1.5_NA$meanCount_round  > 0), "\n")

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 5)

ggplot(boboli_breed_1.5_NA, aes(x = meanCount_round)) +
  geom_histogram(binwidth = 10, boundary = 0, color = "black") +
  labs(x = "Abundance", y = "Frequency") +
  theme_classic(base_size = 16)

## 9. GAM Family & Null Model

Family selection compares Negative Binomial, Zero-Inflated Poisson (ziP), and Poisson on an intercept-only model using AIC. See `wind_analysis_pipeline.ipynb` for full family selection output.

**Result:** Negative Binomial selected.

In [ ]:
m_null <- timed_gam(meanCount_round ~ 1,
                    data = boboli_breed_split$train, family = "nb", knots = time_knots)
summary(m_null)

## 10. Wind Variable Selection

Six wind variables are evaluated against a nuisance-only null model using AICc. All models share the same nuisance structure:

- **Effort:** `s(meanSampDur)`, `s(meanNumObs)`, `s(meanDIST)`, `s(meanStartTime, bs="cc")`
- **Land cover:** `s(Developed)`, `s(Cropland)`, `s(GrassShrub)`, `s(Water)`, `s(Wetland)`, `s(Barren)`, `IceSnow` (linear)
- **Spatial:** `s(Lat)`, `s(Long)`

IceSnow is fit as a linear term due to its near-zero distribution. The shared base formula is extended using `update()` to avoid repeating the nuisance structure.

| Wind variable | Type |
|---------------|------|
| `WindCount`  | continuous smooth |
| `WindPA`     | binary factor |
| `WindHeight` | ordered factor |
| `WindAge`    | ordered factor |
| `WindRSA`    | ordered factor |
| `WindCap`    | ordered factor |

In [ ]:
# Shared nuisance base formula — extended with each wind term via update()
f_nuisance <- meanCount ~ s(meanSampDur, k=k) + s(meanNumObs, k=k) + s(meanDIST, k=k) +
  s(meanStartTime, bs="cc", k=k_time) +
  s(Developed, k=k) + s(Cropland, k=k) + s(GrassShrub, k=k) +
  s(Water, k=k) + s(Wetland, k=k) + s(Barren, k=k) + IceSnow +
  s(Lat, k=k) + s(Long, k=k)

f_wind_count  <- update(f_nuisance, ~ . + s(WindCount, k=k))
f_wind_PA     <- update(f_nuisance, ~ . + factor(WindPA))
f_wind_height <- update(f_nuisance, ~ . + factor(WindHeight))
f_wind_age    <- update(f_nuisance, ~ . + factor(WindAge))
f_wind_rsa    <- update(f_nuisance, ~ . + factor(WindRSA))
f_wind_cap    <- update(f_nuisance, ~ . + factor(WindCap))

In [ ]:
m_wind_count  <- timed_gam(f_wind_count,  boboli_breed_split$train, "nb", time_knots)
m_wind_PA     <- timed_gam(f_wind_PA,     boboli_breed_split$train, "nb", time_knots)
m_wind_height <- timed_gam(f_wind_height, boboli_breed_split$train, "nb", time_knots)
m_wind_age    <- timed_gam(f_wind_age,    boboli_breed_split$train, "nb", time_knots)
m_wind_rsa    <- timed_gam(f_wind_rsa,    boboli_breed_split$train, "nb", time_knots)
m_wind_cap    <- timed_gam(f_wind_cap,    boboli_breed_split$train, "nb", time_knots)
m_wind_null   <- timed_gam(f_nuisance,    boboli_breed_split$train, "nb", time_knots)

In [ ]:
MuMIn::AICc(m_wind_count, m_wind_PA, m_wind_height,
            m_wind_age, m_wind_rsa, m_wind_cap, m_wind_null)

### Wind variable selection results

| Model | AICc | delta AICc vs null |
|-------|------|--------------------|
| `s(WindCount, k=5)` | **~60,425** | **-33** |
| WindPA     | ~60,456 | -2 |
| WindHeight | ~60,458 | ~0 |
| WindRSA    | ~60,460 | +2 |
| WindAge    | ~60,462 | +4 |
| WindCap    | ~60,462 | +4 |
| Null       | ~60,458 | -- |

**Decision:** `s(WindCount, k=5)` selected — provides the largest AICc improvement. All other wind variables perform at or near the null.

## 11. Best Model Summary & Diagnostics

In [ ]:
summary(m_wind_count)

In [ ]:
gam.vcomp(m_wind_count, rescale = TRUE)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 20)
plot(m_wind_count)

## 12. Prediction & Visualization

Predictions are generated on the test set and WindCount is back-transformed to its original scale for plotting.

In [ ]:
wind_mean <- scaling_params$mean[scaling_params$variable == "WindCount"]
wind_sd   <- scaling_params$sd[scaling_params$variable   == "WindCount"]

pred_df <- boboli_breed_split$test %>%
  mutate(
    predicted          = predict.gam(m_wind_count, newdata = boboli_breed_split$test, type = "response"),
    WindCount_original = WindCount * wind_sd + wind_mean
  )

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 6)

ggplot(pred_df, aes(x = WindCount_original, y = predicted)) +
  geom_smooth(color = "#2C7BB6", fill = "#ABD9E9") +
  labs(x = "Number of Wind Turbines", y = "Predicted Abundance",
       title = "Bobolink: Predicted Abundance vs. Wind Turbine Count") +
  theme_bw(base_size = 14)